# Phase 4: Two middle partitions in a chain

Parent runs embedding + first k blocks, then sends hidden states through two chained middle workers, then runs last k blocks + norm + lm_head locally.

Topology: Parent -> Worker 1 -> Worker 2 -> Parent.

In [1]:
import os
import sys
import pickle
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from multiprocessing.shared_memory import SharedMemory

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "Hey! How are you feeling today?"
k = 2
num_middle_partitions = 2
num_new_tokens = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [2]:
base = getattr(model, "model", model)
embed_module = getattr(base, "embed_tokens", None) or getattr(base, "embed", None)
layers_list = getattr(base, "layers", None)
final_norm = getattr(base, "norm", None)
lm_head_module = getattr(model, "lm_head", None)
rotary_emb = getattr(base, "rotary_emb", None)
assert embed_module is not None and layers_list is not None
assert final_norm is not None and lm_head_module is not None and rotary_emb is not None

num_blocks = len(layers_list)
first_k_layers = layers_list[:k]
last_k_layers = layers_list[-k:]
middle_start = k
middle_end = num_blocks - k
middle_count = middle_end - middle_start
assert middle_count > 0
assert num_middle_partitions >= 1
assert middle_count % num_middle_partitions == 0, "middle layers must split evenly"

chunk = middle_count // num_middle_partitions
ranges = []
for i in range(num_middle_partitions):
    s = middle_start + i * chunk
    e = s + chunk
    ranges.append((s, e))

print(f"total blocks={num_blocks}, first_k={k}, middle_count={middle_count}, last_k={k}")
print("middle partition ranges:", ranges)

total blocks=24, first_k=2, middle_count=20, last_k=2
middle partition ranges: [(2, 12), (12, 22)]


In [3]:
inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
input_ids_baseline = inputs["input_ids"].clone()

with torch.no_grad():
    for _ in range(num_new_tokens):
        outputs = model(input_ids=input_ids_baseline, use_cache=False)
        next_token_id = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        input_ids_baseline = torch.cat([input_ids_baseline, next_token_id], dim=-1)

generated_ids_baseline = input_ids_baseline[0, inputs["input_ids"].shape[1]:].tolist()
decoded_baseline = tokenizer.decode(generated_ids_baseline, skip_special_tokens=True)
print("Baseline generated token IDs:", generated_ids_baseline)
print("Baseline decoded:", decoded_baseline)

Baseline generated token IDs: [8713, 498, 8266, 1661, 30, 358, 2776, 537, 2704, 1128, 498, 3076, 553, 330, 18536, 1]
Baseline decoded:  Are you feeling good? I'm not sure what you mean by "good"


In [4]:
HIDDEN_SIZE = model.config.hidden_size
MAX_SEQ = 2048
BATCH_SIZE = 1
max_nbytes = BATCH_SIZE * MAX_SEQ * HIDDEN_SIZE * 4

shm_p_w1 = SharedMemory(create=True, size=max_nbytes)
shm_w1_w2 = SharedMemory(create=True, size=max_nbytes)
shm_w2_p = SharedMemory(create=True, size=max_nbytes)

p_to_w1_r, p_to_w1_w = os.pipe()
w1_to_w2_r, w1_to_w2_w = os.pipe()
w2_to_p_r, w2_to_p_w = os.pipe()

worker_script = os.path.join(os.getcwd(), "middle_worker_v4.py")
assert num_middle_partitions == 2, "this Phase 4 notebook currently instantiates exactly 2 chained workers"

w1 = subprocess.Popen(
    [
        sys.executable,
        worker_script,
        MODEL_NAME,
        str(p_to_w1_r),
        str(w1_to_w2_w),
        shm_p_w1.name,
        shm_w1_w2.name,
        str(max_nbytes),
        str(ranges[0][0]),
        str(ranges[0][1]),
    ],
    pass_fds=[p_to_w1_r, w1_to_w2_w],
)

w2 = subprocess.Popen(
    [
        sys.executable,
        worker_script,
        MODEL_NAME,
        str(w1_to_w2_r),
        str(w2_to_p_w),
        shm_w1_w2.name,
        shm_w2_p.name,
        str(max_nbytes),
        str(ranges[1][0]),
        str(ranges[1][1]),
    ],
    pass_fds=[w1_to_w2_r, w2_to_p_w],
)

os.close(p_to_w1_r)
os.close(w1_to_w2_r)
os.close(w1_to_w2_w)
os.close(w2_to_p_w)

pipe_to_w1 = os.fdopen(p_to_w1_w, "wb")
pipe_from_w2 = os.fdopen(w2_to_p_r, "rb")

def run_layers(hidden_states, layers):
    seq_len = hidden_states.shape[1]
    position_ids = torch.arange(seq_len, device=DEVICE, dtype=torch.long).unsqueeze(0)
    position_embeddings = rotary_emb(hidden_states, position_ids)
    causal_mask = torch.triu(
        torch.full((seq_len, seq_len), torch.finfo(hidden_states.dtype).min, device=DEVICE, dtype=hidden_states.dtype),
        diagonal=1,
    ).unsqueeze(0).unsqueeze(0)
    for layer in layers:
        hidden_states = layer(
            hidden_states,
            attention_mask=causal_mask,
            position_embeddings=position_embeddings,
            position_ids=position_ids,
            use_cache=False,
        )
    return hidden_states

def send_to_chain_receive_from_chain(hidden_states):
    t = hidden_states.cpu().float()
    nbytes = t.numel() * t.element_size()
    if nbytes > max_nbytes:
        raise ValueError(f"nbytes={nbytes} > max_nbytes={max_nbytes}")
    memoryview(shm_p_w1.buf)[:nbytes][:] = t.numpy().tobytes()
    pickle.dump((tuple(t.shape), nbytes), pipe_to_w1)
    pipe_to_w1.flush()

    out_meta = pickle.load(pipe_from_w2)
    if out_meta is None:
        raise RuntimeError("Received unexpected shutdown marker while waiting for output")
    out_shape, out_nbytes = out_meta
    out = torch.frombuffer(memoryview(shm_w2_p.buf)[:out_nbytes], dtype=torch.float32).reshape(out_shape).clone()
    return out.to(DEVICE)

input_ids_split = inputs["input_ids"].clone()
with torch.no_grad():
    for _ in range(num_new_tokens):
        hidden = embed_module(input_ids_split)
        hidden = run_layers(hidden, first_k_layers)
        hidden = send_to_chain_receive_from_chain(hidden)
        hidden = run_layers(hidden, last_k_layers)
        hidden = final_norm(hidden)
        logits_local = lm_head_module(hidden[:, -1:, :])
        next_token_id = logits_local.argmax(dim=-1)
        input_ids_split = torch.cat([input_ids_split, next_token_id], dim=-1)

pickle.dump(None, pipe_to_w1)
pipe_to_w1.flush()
shutdown_meta = pickle.load(pipe_from_w2)
assert shutdown_meta is None

pipe_to_w1.close()
pipe_from_w2.close()
w1.wait()
w2.wait()

shm_p_w1.close()
shm_w1_w2.close()
shm_w2_p.close()
shm_p_w1.unlink()
shm_w1_w2.unlink()
shm_w2_p.unlink()

generated_ids_split = input_ids_split[0, inputs["input_ids"].shape[1]:].tolist()
decoded_split = tokenizer.decode(generated_ids_split, skip_special_tokens=True)
print("Split (Phase 4 chain) generated token IDs:", generated_ids_split)
print("Split decoded:", decoded_split)

Loading weights: 100%|██████████| 290/290 [00:03<00:00, 94.68it/s]


Split (Phase 4 chain) generated token IDs: [8713, 498, 8266, 1661, 30, 358, 2776, 537, 2704, 1128, 498, 3076, 553, 330, 18536, 1]
Split decoded:  Are you feeling good? I'm not sure what you mean by "good"


In [5]:
match = generated_ids_baseline == generated_ids_split
print("Token-by-token comparison:")
for i in range(num_new_tokens):
    status = "ok" if generated_ids_baseline[i] == generated_ids_split[i] else "MISMATCH"
    print(f"  {i}: baseline={generated_ids_baseline[i]}, split={generated_ids_split[i]} [{status}]")
print("\nPASS" if match else "FAIL")
print("\nBaseline decoded:", decoded_baseline)
print("Split decoded:    ", decoded_split)

Token-by-token comparison:
  0: baseline=8713, split=8713 [ok]
  1: baseline=498, split=498 [ok]
  2: baseline=8266, split=8266 [ok]
  3: baseline=1661, split=1661 [ok]
  4: baseline=30, split=30 [ok]
  5: baseline=358, split=358 [ok]
  6: baseline=2776, split=2776 [ok]
  7: baseline=537, split=537 [ok]
  8: baseline=2704, split=2704 [ok]
  9: baseline=1128, split=1128 [ok]
  10: baseline=498, split=498 [ok]
  11: baseline=3076, split=3076 [ok]
  12: baseline=553, split=553 [ok]
  13: baseline=330, split=330 [ok]
  14: baseline=18536, split=18536 [ok]
  15: baseline=1, split=1 [ok]

PASS

Baseline decoded:  Are you feeling good? I'm not sure what you mean by "good"
Split decoded:      Are you feeling good? I'm not sure what you mean by "good"
